#### Following the general framework proposed in Truong et al. (2020) change point detection review, a standard CPD pipeline consists of five key steps:

- step 1: Define the signal
- step 2: Define the type of change
- step 3: Choose the optimization/search method
- step 4: Determine the number of change points (constraints)
- step 5: Evaluate and validate the results

Before entering this pipeline, we first prepare the input data.

In our case, we construct four key performance indicators at the Plate Appearance (PA) level:

* Power Efficiency
* Launch Angle Stability
* Hitting Decisions
* Luck vs. Skill

These features are derived from raw pitch-level data through aggregation and transformation, resulting in a structured PA-level dataset for each player.

In [2]:
### suppose we have the data 
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

In [4]:

data_path = Path("../data/processed/qualified_hitters_statcast_2021_2025_batted_ball.csv")
df = pd.read_csv(data_path)

print(f"Dataset shape: {df.shape}")
print(f"\nColumn names:")
print(df.columns)
display(df.head())

player_counts = df.groupby("Name").size().reset_index(name="num_pitches")
print(f"Total unique players: {player_counts['Name'].nunique()}")
player_counts.sort_values("num_pitches", ascending=False)

Dataset shape: (767599, 64)

Column names:
Index(['Season', 'game_date', 'game_pk', 'batter', 'pitcher', 'player_name',
       'Name', 'IDfg', 'mlbam_id', 'home_team', 'away_team', 'inning',
       'inning_topbot', 'at_bat_number', 'pitch_number', 'balls', 'strikes',
       'stand', 'p_throws', 'pitch_type', 'pitch_name', 'description',
       'events', 'zone', 'plate_x', 'plate_z', 'sz_top', 'sz_bot',
       'launch_speed', 'launch_angle', 'hit_distance_sc',
       'estimated_ba_using_speedangle', 'estimated_woba_using_speedangle',
       'woba_value', 'babip_value', 'iso_value', 'hc_x', 'hc_y', 'bb_type',
       'on_1b', 'on_2b', 'on_3b', 'season', 'batter_name_final', 'is_pitch',
       'is_swing', 'is_whiff', 'is_called_strike', 'is_ball', 'is_in_play',
       'is_zone', 'is_out_of_zone', 'is_batted_ball', 'valid_launch_speed',
       'valid_launch_angle', 'valid_hit_distance_sc', 'valid_xwoba',
       'is_hard_hit', 'is_barrel_proxy', 'exit_velocity',
       'launch_angle_metric',

,Season,game_date,game_pk,batter,pitcher,player_name,Name,IDfg,mlbam_id,home_team,...,valid_launch_angle,valid_hit_distance_sc,valid_xwoba,is_hard_hit,is_barrel_proxy,exit_velocity,launch_angle_metric,xwoba_est,woba_on_play,hit_distance
0,2021,2021-03-15,642094,408234,547943,"Ryu, Hyun Jin",Miguel Cabrera,1744,408234,DET,...,1,1,0,0,0,72.9,70.0,NaN,0.0,127.0
1,2021,2021-03-15,642094,408234,547943,"Ryu, Hyun Jin",Miguel Cabrera,1744,408234,DET,...,1,1,0,0,0,32.4,-36.0,NaN,0.0,2.0
2,2021,2021-03-15,642094,408234,643615,"Zeuch, T.J.",Miguel Cabrera,1744,408234,DET,...,1,1,0,0,0,82.4,-20.0,NaN,NaN,6.0
3,2021,2021-03-15,642094,408234,643615,"Zeuch, T.J.",Miguel Cabrera,1744,408234,DET,...,1,1,0,1,0,96.6,-9.0,NaN,0.9,13.0
4,2021,2021-03-16,642118,408234,543037,"Cole, Gerrit",Miguel Cabrera,1744,408234,DET,...,1,1,0,1,0,103.1,-5.0,NaN,0.0,21.0


Total unique players: 420


,Name,num_pitches
124,Freddie Freeman,5276
214,Jose Ramirez,5214
255,Luis Arraez,5011
276,Matt Olson,4948
268,Marcus Semien,4920
...,...,...
121,Francisco Alvarez,472
299,Mitch Garver,456
195,Joey Wiemer,454
106,Edouard Julien,452


### step 1: Define the Signal

According to Truong et al. (2020) change point detection review, the first step in CPD is to define the signal, including the time index and preprocessing method.

In our setting, the signal is not raw pitch-level data, but a PA-level feature time series constructed from the four indicators above.

Decision

Time index: PA order
Signal: rolling feature (50–80 PA window)

This produces a smoothed univariate time series for each player and each feature, which serves as the input for CPD modeling.